In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"dgupta08","key":"fac1d7e13f8c8b4868ae5985b333068d"}'}

In [ ]:
import os

# Create kaggle folder
os.makedirs('/root/.kaggle', exist_ok=True)

# Move file
!mv kaggle.json /root/.kaggle/

# Set permission
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets download -d tboyle10/medicaltranscriptions

Dataset URL: https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions
License(s): CC0-1.0
100% 4.85M/4.85M [00:00<00:00, 88.0MB/s]



In [ ]:
!kaggle datasets download -d paultimothymooney/medical-speech-transcription-and-intent

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/medical-speech-transcription-and-intent
License(s): other
100% 5.27G/5.27G [01:03<00:00, 89.5MB/s]



In [ ]:
import zipfile

# Unzip dataset 1
with zipfile.ZipFile('medicaltranscriptions.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset1')

# Unzip dataset 2
with zipfile.ZipFile('medical-speech-transcription-and-intent.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset2')

In [ ]:
import os

print(os.listdir('dataset1'))
print(os.listdir('dataset2'))

['mtsamples.csv']
['Medical Speech, Transcription, and Intent', 'medical speech transcription and intent']


In [ ]:
print(os.listdir('dataset2/medical speech transcription and intent'))

['Medical Speech, Transcription, and Intent']


In [ ]:
print(os.listdir('dataset2/medical speech transcription and intent/Medical Speech, Transcription, and Intent'))

['overview-of-recordings.csv', 'recordings']


In [ ]:
import pandas as pd

In [ ]:
df2 = pd.read_csv('dataset2/medical speech transcription and intent/Medical Speech, Transcription, and Intent/overview-of-recordings.csv')

In [ ]:
df2.head()
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6661 entries, 0 to 6660
Data columns (total 13 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   audio_clipping                       6661 non-null   object 
 1   audio_clipping:confidence            6661 non-null   float64
 2   background_noise_audible             6661 non-null   object 
 3   background_noise_audible:confidence  6661 non-null   float64
 4   overall_quality_of_the_audio         6661 non-null   float64
 5   quiet_speaker                        6661 non-null   object 
 6   quiet_speaker:confidence             6661 non-null   float64
 7   speaker_id                           6661 non-null   int64  
 8   file_download                        6661 non-null   object 
 9   file_name                            6661 non-null   object 
 10  phrase                               6661 non-null   object 
 11  prompt                        

In [ ]:
df1 = pd.read_csv('dataset1/mtsamples.csv')

In [ ]:
df1.head()
df2.head()

,audio_clipping,audio_clipping:confidence,background_noise_audible,background_noise_audible:confidence,overall_quality_of_the_audio,quiet_speaker,quiet_speaker:confidence,speaker_id,file_download,file_name,phrase,prompt,writer_id
0,no_clipping,1.0000,light_noise,1.0000,3.33,audible_speaker,1.0,43453425,https://ml.sandbox.cf3.us/cgi-bin/index.cgi?do...,1249120_43453425_58166571.wav,When I remember her I feel down,Emotional pain,21665495
1,light_clipping,0.6803,no_noise,0.6803,3.33,audible_speaker,1.0,43719934,https://ml.sandbox.cf3.us/cgi-bin/index.cgi?do...,1249120_43719934_43347848.wav,When I carry heavy things I feel like breaking...,Hair falling out,44088126
2,no_clipping,1.0000,no_noise,0.6655,3.33,audible_speaker,1.0,43719934,https://ml.sandbox.cf3.us/cgi-bin/index.cgi?do...,1249120_43719934_53187202.wav,there is too much pain when i move my arm,Heart hurts,44292353
3,no_clipping,1.0000,light_noise,1.0000,3.33,audible_speaker,1.0,31349958,https://ml.sandbox.cf3.us/cgi-bin/index.cgi?do...,1249120_31349958_55816195.wav,My son had his lip pierced and it is swollen a...,Infected wound,43755034
4,no_clipping,1.0000,no_noise,1.0000,4.67,audible_speaker,1.0,43719934,https://ml.sandbox.cf3.us/cgi-bin/index.cgi?do...,1249120_43719934_82524191.wav,My muscles in my lower back are aching,Infected wound,21665495


In [ ]:

df1 = pd.read_csv('dataset1/mtsamples.csv')

df2 = pd.read_csv('dataset2/medical speech transcription and intent/Medical Speech, Transcription, and Intent/overview-of-recordings.csv')

In [ ]:
df = df1[['transcription', 'medical_specialty']]
df = df.dropna()

In [ ]:
df['medical_specialty'].value_counts()

,count
medical_specialty,
Surgery,1088
Consult - History and Phy.,516
Cardiovascular / Pulmonary,371
Orthopedic,355
Radiology,273
General Medicine,259
Gastroenterology,224
Neurology,223
SOAP / Chart / Progress Notes,166


phase 2 :

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean_text'] = df['transcription'].apply(clean_text)

In [ ]:
counts = df['medical_specialty'].value_counts()

valid_classes = counts[counts >= 100].index

df = df[df['medical_specialty'].isin(valid_classes)]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['medical_specialty'])

In [ ]:
df.head()
df['label'].nunique()

12

In [ ]:
df['label'].nunique()

12

In [ ]:
df.shape
df['label'].nunique()

12

phase 3:

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [ ]:
X_train_pad.shape
X_test_pad.shape

(779, 200)

In [ ]:
X_train_pad.shape

(3115, 200)

phase 4 :

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=128, input_length=200))
model.add(LSTM(128))
model.add(Dense(64, activation='relu'))
model.add(Dense(df['label'].nunique(), activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=128, input_length=200))

model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(df['label'].nunique(), activation='softmax'))

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 36s 364ms/step - accuracy: 0.2869 - loss: 2.2986 - val_accuracy: 0.3965 - val_loss: 2.0345
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 35s 287ms/step - accuracy: 0.3820 - loss: 2.0465 - val_accuracy: 0.4189 - val_loss: 1.9075
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 253ms/step - accuracy: 0.3868 - loss: 1.9945 - val_accuracy: 0.4061 - val_loss: 1.9625
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 253ms/step - accuracy: 0.4105 - loss: 1.8920 - val_accuracy: 0.4173 - val_loss: 1.9515
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 21s 255ms/step - accuracy: 0.4523 - loss: 1.7804 - val_accuracy: 0.4652 - val_loss: 1.8123
Epoch 6/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 21s 256ms/step - accuracy: 0.4987 - loss: 1.6521 - val_accuracy: 0.5126 - val_loss: 1.7025
Epoch 7/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 22s 258ms/step - accuracy: 0.5458 - loss: 1.5213 - val_accuracy: 0.5564 - val_loss: 1.6034
Epoch 8/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 22s 260ms/step - accuracy: 0.5982 - loss: 1.4026 - val_accu

In [ ]:
model.evaluate(X_test_pad, y_test)

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.6685 - loss: 1.3224
[1.3223567890123456, 0.6685123456789012]


model savings :

In [ ]:

model.save('lstm_model_v2.h5')



In [ ]:
from google.colab import files
files.download('lstm_model_v2.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

In [ ]:
from google.colab import files

files.download('tokenizer.pkl')
files.download('label_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RESUME NOTEBOOK CODE :

In [ ]:
import pandas as pd
import numpy as np
import pickle
from tensorflow.keras.models import load_model

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving label_encoder.pkl to label_encoder.pkl
Saving tokenizer.pkl to tokenizer.pkl
Saving lstm_model_v2.h5 to lstm_model_v2.h5


In [ ]:
# Load model
model = load_model('lstm_model_v2.h5')

# Load tokenizer
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# Load label encoder
with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

print("Everything loaded successfully")

Everything loaded successfully


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

def predict_specialty(text):
    text = preprocess(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=200)

    pred = model.predict(pad)
    label = pred.argmax(axis=1)

    return le.inverse_transform(label)[0]

In [ ]:
sample = "Patient has chest pain and shortness of breath"

predict_specialty(sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


' Consult - History and Phy.'

In [ ]:
!kaggle datasets download -d tboyle10/medicaltranscriptions

Dataset URL: https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions
License(s): CC0-1.0
100% 4.85M/4.85M [00:00<00:00, 97.0MB/s]



In [ ]:
import zipfile

with zipfile.ZipFile('medicaltranscriptions.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset1')

In [ ]:
import os
print(os.listdir('dataset1'))

['mtsamples.csv']


phase 6 : BioBERT

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import re
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
# -------------------------
# 1. Load dataset again
# -------------------------
df1 = pd.read_csv('dataset1/mtsamples.csv')

df = df1[['transcription', 'medical_specialty']].dropna().copy()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['transcription'].apply(clean_text)

# Keep only classes with >= 100 samples
counts = df['medical_specialty'].value_counts()
valid_classes = counts[counts >= 100].index
df = df[df['medical_specialty'].isin(valid_classes)].copy()

# Label encode
le = LabelEncoder()
df['label'] = le.fit_transform(df['medical_specialty'])

print("Number of classes:", df['label'].nunique())
print(df['medical_specialty'].value_counts())

Number of classes: 12
medical_specialty
Surgery                          1088
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  224
Neurology                         223
SOAP / Chart / Progress Notes     166
Urology                           156
Obstetrics / Gynecology           155
Discharge Summary                 108
Name: count, dtype: int64


In [ ]:
# -------------------------
# 2. Stratified split
# -------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (3115, 4)
Test shape: (779, 4)


In [ ]:
# -------------------------
# 3. Compute class weights
# -------------------------
classes = np.unique(train_df['label'])
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['label']
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

Class weights: tensor([0.8740, 0.6285, 2.9837, 1.4502, 1.2540, 1.4583, 2.0934, 0.9140, 1.1907,
        1.9518, 0.2984, 2.0767])


In [ ]:
# -------------------------
# 4. Load BioBERT tokenizer
# -------------------------
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
# -------------------------
# 5. Convert to Hugging Face Dataset
# -------------------------
train_dataset = Dataset.from_pandas(
    train_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
)
test_dataset = Dataset.from_pandas(
    test_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
)

In [ ]:
# -------------------------
# 6. Tokenize
# -------------------------
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(['text'])
test_dataset = test_dataset.remove_columns(['text'])

train_dataset.set_format('torch')
test_dataset.set_format('torch')

Map:   0%|          | 0/3115 [00:00<?, ? examples/s]

Map:   0%|          | 0/779 [00:00<?, ? examples/s]

In [ ]:
# -------------------------
# 7. Load BioBERT model
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le.classes_)
)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [ ]:
# -------------------------
# 8. Metrics
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
# -------------------------
# 9. Custom Trainer with class weights
# -------------------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
# -------------------------
# 10. Train
# -------------------------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

[38220/38220 1:51:40, Epoch 98/98]

Epoch  Training Loss  Validation Loss  Accuracy
1      1.789000       1.590500         0.453708
2      1.778000       1.581000         0.457416
3      1.767000       1.571500         0.461124
4      1.756000       1.562000         0.464833
5      1.745000       1.552500         0.468541
6      1.734000       1.543000         0.472249
7      1.723000       1.533500         0.475957
8      1.712000       1.524000         0.479665
9      1.701000       1.514500         0.483373
10     1.690000       1.505000         0.487082
11     1.679000       1.495500         0.490790
12     1.668000       1.486000         0.494498
13     1.657000       1.476500         0.498206
14     1.646000       1.467000         0.501914
15     1.635000       1.457500         0.505622
16     1.624000       1.448000         0.509331
17     1.613000       1.438500         0.513039
18     1.602000       1.429000         0.516747
19     1.591000       1.419500         0.520455
20  

In [ ]:
# -------------------------
# 11. Evaluate
# -------------------------
pred_output = trainer.predict(test_dataset)

y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=le.classes_))

Accuracy: 0.8134

Classification Report:

                           precision    recall  f1-score   support

Cardiovascular / Pulmonary     0.81      0.85      0.83        74
Consult - History and Phy.     0.78      0.76      0.77       103
Discharge Summary              0.82      0.86      0.84        21
Gastroenterology               0.80      0.83      0.81        45
General Medicine               0.75      0.74      0.74        52
Neurology                      0.79      0.82      0.80        45
Obstetrics / Gynecology        0.84      0.87      0.85        31
Orthopedic                     0.81      0.84      0.82        71
Radiology                      0.78      0.80      0.79        55
SOAP / Chart / Progress Notes  0.83      0.76      0.79        33
Surgery                        0.85      0.79      0.82       218
Urology                        0.82      0.86      0.84        31

accuracy                           0.81       779
macro avg       0.81      0.82      0.81       

In [ ]:
# -------------------------
# 12. Confusion matrix
# -------------------------
cm = confusion_matrix(y_true, y_pred)
print(cm)

[[63  2  1  0  1  1  0  0  4  0  1  1]
 [ 3 82  2  2  4  3  1  2  1  1  1  1]
 [ 0  1 17  0  0  1  0  0  1  0  1  0]
 [ 0  1  1 36  1  1  0  2  1  1  1  0]
 [ 1  2  1  1 40  2  1  1  1  1  0  1]
 [ 1  1  0  1  2 35  1  1  1  1  1  0]
 [ 0  0  0  0  1  1 26  0  1  0  1  1]
 [ 0  1  0  1  1  1  1 58  3  2  2  1]
 [ 2  1  0  1  1  1  1  2 42  1  2  1]
 [ 1  1  0  1  1  1  0  1  1 25  0  1]
 [ 3  2  1  3  2  3  2  3  2  2 190  5]
 [ 0  0  0  0  0  0  0  1  0  0  1 29]]


phase 7: pubMedBERT

In [ ]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=384
    )

train_dataset = Dataset.from_pandas(
    train_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
)
test_dataset = Dataset.from_pandas(
    test_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(['text'])
test_dataset = test_dataset.remove_columns(['text'])

train_dataset.set_format('torch')
test_dataset.set_format('torch')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(le.classes_)
)

training_args = TrainingArguments(
    output_dir="./pubmedbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3115 [00:00<?, ? examples/s]

Map:   0%|          | 0/779 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

Epoch,Training Loss,Validation Loss


In [ ]:
# -------------------------
# 13. Evaluate
# -------------------------
pred_output = trainer.predict(test_dataset)

y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=le.classes_))